# 03 - Error analysis of the final blend

Where does the final blend (run 20261031T220901Z_1ff742f2, LightGBM 2/6 + CatBoost 2/6 + ExtraTrees 1/6 + XGBoost 1/6) rank well and badly, and what does its main member use?

The code lives in reports/figs/make_figures.py, so the report's tables and figures and this notebook come from the same functions. Every step reads logged out-of-fold predictions, except the importance step, which refits the tuned LightGBM on folds 1 to 4 (about 1.5 minutes).

In [ ]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Image, Markdown, display

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "reports" / "figs"))
import make_figures as mf

FIGS = mf.FIGS

## 1. Slices

Seven slicings, fixed before looking: the 15 largest states, loan product, aus-1, income quintile, reported LTV band, exempt record (code 1111 in any coded column), and the pattern of blanks in LTV, property value, DTI, income, loan term and location.

Choices made before the run:

- A level under 1,000 rows is listed but not read: below that, a level holds ~150 denials or fewer and its AUC interval is wider than most differences between levels.
- AUC inside a slice ranks rows of that slice only. It is usually below the pooled AUC, because telling slices apart no longer counts, so levels are compared with each other, not with the pooled value.
- Residual = accept rate - mean prediction, with the binomial SE of the accept rate. It checks calibration inside each slice; AUC ignores calibration, so a residual says nothing about the score.
- Intervals: 1,000-resample stratified bootstrap (positives and negatives resampled apart), as everywhere in the project.

In [ ]:
mf.slices()
display(Markdown((FIGS / "t10_slices.md").read_text()))

In [ ]:
display(Image(FIGS / "f7_slices.png", width=650))

### Reading

- Calibration: none of the 44 levels read has a residual beyond 2 SE (largest: Florida -0.008 and New York -0.012, both 1.95 SE). One or two near 2 SE out of 44 is what chance gives.
- Exempt records rank worst: 0.773 against 0.923 for the others. The blank pattern LTV + property value + DTI + loan term is 99.6% exempt rows and scores 0.764.
- The blank pattern DTI + income (66% non-natural-person applicants) scores 0.793.
- FHA and VA first liens rank at 0.850 against 0.937 for conventional first liens; aus-1 = TOTAL, the FHA scorecard, at 0.842. We think the DTI rank separates less inside FHA and VA; tested in section 4.
- AUC falls with income (0.947 in the lowest quintile, 0.860 in the highest) while the denied share falls from 32% to 8%.
- States: 0.879 to 0.937, intervals overlapping; no state stands apart.

=> The worst-error inspection starts from three groups: exempt records, non-natural-person applicants, government loans.

In [ ]:
# check behind two of the statements above: which rows make up the two weak blank patterns
train, _ = mf.load()
pattern = mf.slice_labels(train)["blank pattern (LTV, property value, DTI, income, term, location)"]
exempt = (train[mf.CODED].astype("string") == "1111").any(axis=1)
entity = train.applicant_sex.astype("string") == "4"
pd.DataFrame({p: {"rows": (pattern == p).sum(), "exempt share": exempt[pattern == p].mean(),
                  "entity share": entity[pattern == p].mean()}
              for p in ["LTV + property value + DTI + loan term", "DTI + income"]}).T.round(3)

## 2. What the main member uses

Importance needs a model and only predictions are logged. The tuned LightGBM (weight 1/3 of the blend) is refitted on folds 1 to 4 with its logged settings; the step stops unless the refit reproduces the logged fold-0 AUC to 1e-6, so it is the logged fold model. It is explained on 10,000 rows of fold 0, drawn with the class balance kept.

- Permutation importance: AUC lost when a column is shuffled, 5 shuffles. Also whole blocks shuffled at once (finance, process, geography and census, demographics), with the same permutation inside a block, because correlated columns share credit.
- SHAP: LightGBM's exact TreeSHAP (pred_contrib), in log-odds; the step checks that they add up to each row's raw score.

In [ ]:
mf.shap()
imp = pd.read_csv(FIGS / "f8_importance.csv")
display(pd.read_csv(FIGS / "f8_blocks.csv").round(4))
imp.head(20).round(4)

In [ ]:
display(Image(FIGS / "f8_permutation.png", width=800))
display(Image(FIGS / "f9_shap.png", width=650))

### Reading

- The DTI rank dominates: shuffling it costs 0.083 AUC, three times the next column; the highest DTI levels lower the log-odds by 3 to 5.
- Then the two leak columns (initially_payable_to_institution 0.029, preapproval 0.022) and aus-1 (0.019). preapproval = 1 adds +4 to +5 to the log-odds, initially_payable_to_institution = 3 removes 6 to 8.5.
- By block: finance 0.160, process 0.128, demographics 0.009, geography and census 0.009.
- construction_method: 5th largest mean |SHAP| (0.17), but 0.0015 when shuffled alone. We think the manufactured-home columns repeat it; tested in section 4. Single-column permutation understates correlated columns; read the blocks for them.
- Shuffling the demographic block costs 0.009 for a model trained with it. Retraining without it is a different measurement: the no-demographics ablation (0.0027).
- These are attributions for this model. They say what moves its score, not what makes a lender accept.

## 3. The worst errors

The 200 worst false positives (denied rows with the highest scores) and the 200 worst false negatives (accepted rows with the lowest), read by hand first, then compared with the rows of the other class that got the same scores. A difference there is something the model saw and still could not use; a difference against all rows of the class would mostly repeat what the model already knows.

Each candidate feature that the reading suggests is then screened on all rows: the residual (accepted - prediction) with the candidate minus the residual without it, inside the regions where the errors sit. A tree can only gain from a column where it leaves such a difference.

In [ ]:
mf.worst_errors()
display(Markdown((FIGS / "t11_worst_errors.md").read_text()))

In [ ]:
# the rows themselves, for reading by hand
rows = pd.read_csv(FIGS / "t11_worst_rows.csv")
cols = ["error", "pred", "loan_type", "construction_method", "aus-1", "debt_to_income_ratio", "loan_to_value_ratio",
        "property_value", "income", "loan_term", "applicant_credit_score_type", "submission_of_application",
        "initially_payable_to_institution", "state_code"]
display(rows[rows.error == "false positive"][cols].head(25))
rows[rows.error == "false negative"][cols].head(25)

### Reading

- Worst false positives: conventional first liens on site-built homes, owner-occupied, DU or LP, moderate DTI, 360 months. Against accepted rows with the same scores they differ only a little (fewer submitted directly, more exempt records).
- They hold more single male applicants and more applicants over 74: protected attributes, not features; noted for the fairness section.
- Worst false negatives: manufactured homes, no automated underwriting, DTI above 60%, where acceptance is rare and the blend is calibrated.
- Dropped: the application channel (calibrated in every combination) and a county rate on exempt rows (lowers their AUC).

=> Three buildable features for the models side, chosen after looking, so the CV decides:

- H1 term_off_grid: loan_term not a multiple of 12. On manufactured homes the residual is 0.073 higher with it (z 3.6), the only candidate past a Bonferroni threshold.
- H2 ltv_blank_value_present: combined LTV blank while the property value is reported. Residual 0.010 higher among scores above 0.95 (z 2.7).
- H3 computed_ltv_above_100: loan amount above property value. Weakest: z 1.9 and 0.9, and the sign flips on manufactured homes alone.

## 4. Tests of the explanations

Each explanation used in the report is stated as a hypothesis, tested here on the data, and reported as held or not; an explanation that was not tested stays out of the report.

- We think the DTI rank, the strongest column, separates accepted from denied less well inside FHA and VA than inside conventional loans.
- We think construction_method scores low in single-column permutation because the manufactured-home columns repeat it.

In [ ]:
mf.checks()
display(Markdown((FIGS / "checks.md").read_text()))

### Result

- DTI: AUC of the DTI rank alone 0.680 for conventional first liens, 0.668 for FHA, 0.631 for VA. Held for VA, barely for FHA; the FHA ranking gap of the blend (0.850 against 0.937) stays unexplained.
- construction_method: every site-built row is "not applicable" or exempt in both manufactured-home columns, and "not applicable" appears on only 16 and 115 other rows. Held.